# 📡 Exercise 01 — Fetch OSM Data

**Day 1 · Guided**

Fetch OpenStreetMap amenity data for a district in your city, inspect the GeoDataFrame, extract coordinates, and save locally.

---

### What is OSM?

OpenStreetMap is a free, community-maintained map of the world. No enforced schema — tags are conventions, not standards.

### What is OSMnx?

[OSMnx](https://osmnx.readthedocs.io/) wraps the Overpass API and returns clean GeoDataFrames. Explore data interactively at [overpass-turbo.eu](https://overpass-turbo.eu).

### 1.1 🏘️ Pick your city **and district**

Look up your city in [`docs/cities.md`](../docs/cities.md) and choose a **district / neighbourhood** within it. Fetching a whole city returns too much data and overloads the API.

> Use `"District, City, Country"` — e.g. `"Södermalm, Stockholm, Sweden"`, `"el Poblenou, Barcelona, Spain"`, `"Kreuzberg, Berlin, Germany"`. Verify on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) that the name resolves.

In [4]:
# TODO: Replace with your district + city + country
#PLACE = "Your District, Your City, Country"  # e.g. "Södermalm, Stockholm, Sweden"
PLACE = "Gràcia, Barcelona, Spain"

print(f"Working with: {PLACE}")

Working with: Gràcia, Barcelona, Spain


### 1.2 📥 Fetch amenities

We pass a curated list of amenity types (not *all* amenities) to keep the request small.

In [5]:
import osmnx as ox

AMENITY_TYPES = [
    "restaurant", "cafe", "fast_food", "bar", "pub",
    "pharmacy", "hospital", "clinic", "dentist",
    "school", "university", "library", "kindergarten",
    "bank", "atm", "post_office",
]

print(f"Fetching amenity data for {PLACE}...")
gdf = ox.features_from_place(PLACE, tags={"amenity": AMENITY_TYPES})
gdf = gdf.reset_index()

print(f"Fetched {len(gdf)} features · {len(gdf.columns)} columns")
print(f"Geometry types:\n{gdf.geometry.geom_type.value_counts()}")

Fetching amenity data for Gràcia, Barcelona, Spain...
Fetched 939 features · 196 columns
Geometry types:
Point      892
Polygon     47
Name: count, dtype: int64


#### Troubleshooting

| Error | Fix |
|-------|-----|
| `InsufficientResponseError` | Check spelling — use `"District, City, Country"` format |
| `ConnectionError` / timeout | Wait 30 s, retry |
| 0 features | Verify name on [nominatim.openstreetmap.org](https://nominatim.openstreetmap.org/) |
| Very slow | Pick a smaller district or reduce `AMENITY_TYPES` |

> ⚠️ The Overpass API is a free shared resource. Don't spam requests — save locally and reload.

### 1.3 🔎 Inspect the raw data

In [14]:
# Look at one feature — all its columns and values
gdf.iloc[2]

element                                                node
id                                                 82781676
geometry                       POINT (2.1553363 41.4039465)
addr:city                                         Barcelona
addr:full                               carrer Asturies, 54
                                           ...             
panoramax:1                                             NaN
member_of                                               NaN
building:levels:underground                             NaN
fax                                                     NaN
type                                                    NaN
Name: 2, Length: 196, dtype: object

In [ ]:
# TODO: Look at a few more features — are they all structured the same way?
# Try: gdf.iloc[1], gdf.iloc[-1]
# What columns are always present? Which ones are mostly NaN?

**Think:** Which columns does every feature have? Why are most columns mostly NaN?

### 1.4 🌐 Extract coordinates → DataFrame

We need plain `lat`/`lon` columns. `geometry.centroid` works for both Points and Polygons.

In [15]:
import pandas as pd

# Extract lat/lon from geometry centroids (works for Points and Polygons)
gdf["lat"] = gdf.geometry.centroid.y
gdf["lon"] = gdf.geometry.centroid.x

# Convert to regular DataFrame (drop geometry column)
df = pd.DataFrame(gdf.drop(columns=["geometry"]))

# Rename osmid → id for consistency
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})

# Drop osmnx metadata columns
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Shape: {df.shape}  (rows, columns)")
print(f"Columns: {len(df.columns)}")
df.head()

Shape: (939, 197)  (rows, columns)
Columns: 197


C:\Users\Lakzhmy\AppData\Local\Temp\ipykernel_676\4124565904.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y
C:\Users\Lakzhmy\AppData\Local\Temp\ipykernel_676\4124565904.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x


,element,id,addr:city,addr:full,addr:housenumber,addr:postcode,addr:street,amenity,check_date,contact:phone,...,panoramax,panoramax:0,architect,panoramax:1,member_of,building:levels:underground,fax,type,lat,lon
0,node,82753755,Barcelona,"carrer Ramon I Cajal, 24",24,08012,Carrer de Ramón y Cajal,pharmacy,2024-11-14,+34932135817,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.402512,2.158573
1,node,82781675,Barcelona,"carrer Verdi, 7",7,08012,Carrer de Verdi,pharmacy,2024-03-24,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.403115,2.157597
2,node,82781676,Barcelona,"carrer Asturies, 54",54,08012,Carrer d'Astúries,pharmacy,2023-11-10,+34932186196,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.403947,2.155336
3,node,430755048,Barcelona,NaN,252,08023,Avinguda de Vallcarca,pharmacy,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.419799,2.140857
4,node,430755135,Barcelona,"avinguda Vallcarca, 188",188,08023,Avinguda de Vallcarca,pharmacy,NaN,+34932843678,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.415487,2.142485


In [18]:
# OSMnx already expanded tags into separate columns — let's see what we have
meta_cols = {"element_type", "id", "lat", "lon"}
tag_cols = sorted([c for c in df.columns if c not in meta_cols])
print(f"Number of tag columns: {len(tag_cols)}")

# TODO: Print the first 20 tag column names
# Hint: tag_cols[:20]
tag_cols[:20]
#print(tag_cols[:20])

Number of tag columns: 194


['access',
 'addr:city',
 'addr:floor',
 'addr:full',
 'addr:housename',
 'addr:housenumber',
 'addr:place',
 'addr:postcode',
 'addr:street',
 'air_conditioning',
 'alt_name',
 'alt_name:ca',
 'amenity',
 'architect',
 'atm',
 'atm:network',
 'atm:operator',
 'bar',
 'billiards',
 'board_games']

In [ ]:
# Verify key columns are present
key_cols = ["id", "lat", "lon", "amenity", "name", "cuisine", "opening_hours"]
for col in key_cols:
    present = col in df.columns
    filled = df[col].notna().sum() if present else 0
    print(f"  {col}: {'✓' if present else '✗'}  ({filled} non-null)")

print(f"\nTotal shape: {df.shape}")

### 1.5 💾 Save locally

In [ ]:
import os
os.makedirs("../data", exist_ok=True)

# Derive a slug from the place name for file naming
city_slug = PLACE.split(",")[0].strip().lower().replace(" ", "_")
gdf.to_file(f"../data/raw_{city_slug}.gpkg", driver="GPKG")

print(f"Saved to ../data/raw_{city_slug}.gpkg")
print(f"Size: {os.path.getsize(f'../data/raw_{city_slug}.gpkg') / 1024:.0f} KB")

### 1.6 👀 First look

In [ ]:
# TODO: Run these commands and study the output

# 1. What are the dtypes?
# df.dtypes

# 2. How much data is missing per column?
# df.info()

# 3. What are the most common amenity types in your city?
# df["amenity"].value_counts().head(10)

---

✅ Raw GeoPackage in `data/` · ✅ Flat DataFrame with lat/lon · ✅ First look at the data

**Next →** [Exercise 02 — Explore & Clean](02_explore_and_clean.ipynb)